# DiffCSE on SICKR

Official paper: https://aclanthology.org/2022.naacl-main.311/  
Official code: https://github.com/voidism/DiffCSE  
Checkpoint: `voidism/diffcse-bert-base-uncased-sts`  
Dataset source: verypluming/JSICK at commit b3034994192fae2f41b5937bcf69544e4282fc39 (original English columns, test split) (https://aclanthology.org/L14-1314/)

This notebook **re-evaluates the authors' released pretrained checkpoint** on one public test set. It does not retrain the original method. It uses a common direct-cosine protocol: encode each sentence independently, compute pairwise cosine similarity, then compare with human scores using Spearman and Pearson correlation. It requires an NVIDIA CUDA GPU and keeps all code in this notebook. Its outputs are empty until you run all cells on the RTX 3080.


In [1]:
# Install only missing packages; preserve an existing CUDA-enabled PyTorch build.
import importlib.util
import subprocess
import sys

requirements = {
    "torch": "torch>=2.2,<3",
    "transformers": "transformers==4.46.3",
    "datasets": "datasets==3.2.0",
    "huggingface_hub": "huggingface_hub>=0.26,<1",
    "numpy": "numpy>=1.26,<3",
    "scipy": "scipy>=1.11,<2",
    "pandas": "pandas>=2.1,<3",
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import importlib.metadata as metadata
for module in requirements:
    package = "huggingface-hub" if module == "huggingface_hub" else module
    print(f"{package}=={metadata.version(package)}")


torch==2.1.2+cu118
transformers==4.46.3
datasets==3.2.0
huggingface-hub==0.36.2
numpy==1.23.5
scipy==1.15.3
pandas==2.3.3


In [2]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for a CUDA GPU. Check the RTX 3080 driver and CUDA-enabled PyTorch installation.")
DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Batch size 32 and dynamic padding fit typical 10/12 GiB RTX 3080 cards.
BATCH_SIZE = 32
MAX_LENGTH = 512


GPU: NVIDIA GeForce RTX 3080 Ti
CUDA runtime: 11.8
GPU memory (GiB): 11.63


In [3]:
METHOD_NAME = 'DiffCSE'
DATASET_NAME = 'SICKR'
MODEL_ID = 'voidism/diffcse-bert-base-uncased-sts'
POOLING = 'cls_before_pooler'
EXPECTED_TEST_ROWS = 4927
LABEL_RANGE = (1.0, 5.0)


In [4]:
if DATASET_NAME == "STSB":
    raw = load_dataset("mteb/stsbenchmark-sts", split="test")
    frame = pd.DataFrame({
        "sentence1": raw["sentence1"],
        "sentence2": raw["sentence2"],
        "gold_score": raw["score"],
    })
else:
    # Read the original English SICK columns from the JSICK source at a fixed Git commit.
    # JSICK preserves the standard train/trial/test labels. No remote dataset code runs.
    import hashlib
    import io
    import os
    import tempfile
    from pathlib import Path
    SICK_COMMIT = "b3034994192fae2f41b5937bcf69544e4282fc39"
    SICK_SHA256 = "5e2ec0dbbb1030eb45bddc817dcda8d53da04ddae817456f89c772905c4f6c8a"
    sick_cache_path = os.environ.get("SICK_TSV_PATH")
    if sick_cache_path:
        sick_tsv = Path(sick_cache_path).read_bytes()
    else:
        with tempfile.TemporaryDirectory(prefix="sick_source_") as source_dir:
            subprocess.run(["git", "-C", source_dir, "init", "-q"], check=True)
            subprocess.run(["git", "-C", source_dir, "fetch", "--depth", "1",
                            "https://github.com/verypluming/JSICK.git", SICK_COMMIT],
                           check=True, capture_output=True, text=True)
            sick_tsv = subprocess.check_output(["git", "-C", source_dir, "show",
                                                f"{SICK_COMMIT}:jsick/jsick.tsv"])
    assert hashlib.sha256(sick_tsv).hexdigest() == SICK_SHA256, "Unexpected JSICK source file"
    raw = pd.read_table(io.BytesIO(sick_tsv))
    raw = raw.loc[raw["data"].eq("test")].copy()
    frame = pd.DataFrame({
        "sentence1": raw["sentence_A_En"].to_numpy(),
        "sentence2": raw["sentence_B_En"].to_numpy(),
        "gold_score": raw["relatedness_score_En"].to_numpy(),
    })

assert len(frame) == EXPECTED_TEST_ROWS, f"Unexpected test set size: {len(frame)}"
assert frame.notna().all().all(), "Dataset contains missing values"
assert frame["gold_score"].between(*LABEL_RANGE).all(), "Score outside documented range"
assert frame["sentence1"].str.len().gt(0).all()
assert frame["sentence2"].str.len().gt(0).all()
print("Dataset:", DATASET_NAME, "| test pairs:", len(frame))
print("Gold score range:", (frame.gold_score.min(), frame.gold_score.max()))
display(frame.head(5))


Dataset: SICKR | test pairs: 4927
Gold score range: (1.0, 5.0)


,sentence1,sentence2,gold_score
0,There is no boy playing outdoors and there is ...,A group of kids is playing in a yard and an ol...,3.3
1,A group of boys in a yard is playing and a man...,The young boys are playing outdoors and the ma...,3.7
2,A group of children is playing in the house an...,The young boys are playing outdoors and the ma...,3.0
3,A brown dog is attacking another animal in fro...,A brown dog is attacking another animal in fro...,4.9
4,A brown dog is attacking another animal in fro...,A brown dog is helping another animal in front...,3.7


In [5]:
api = HfApi()
model_sha = api.model_info(MODEL_ID).sha
dataset_repo = "verypluming/JSICK"
dataset_sha = SICK_COMMIT
print("Model:", MODEL_ID, "| revision:", model_sha)
print("Dataset:", dataset_repo, "| revision:", dataset_sha)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_sha)
model = AutoModel.from_pretrained(MODEL_ID, revision=model_sha).to(DEVICE)
model.eval()
assert model.config.hidden_size == 768, "Expected a BERT-base-sized checkpoint"


Model: voidism/diffcse-bert-base-uncased-sts | revision: ea605f716bd57f97407640374487c2c9effc94af
Dataset: verypluming/JSICK | revision: b3034994192fae2f41b5937bcf69544e4282fc39


/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Some weights of the model checkpoint at voidism/diffcse-bert-base-uncased-sts were not used when initializing BertModel: ['aux_bert.embeddings.LayerNorm.bias', 'aux_bert.embeddings.LayerNorm.weight', 'aux_bert.embeddings.position_embeddings.weight', 'aux_bert.embeddings.position_ids', 'aux_bert.embeddings.token_type_embeddings.weight', 'aux_bert.embeddings.word_embeddings.weight', 'aux_bert.encoder.layer.0.attention.output.LayerNorm.bias', 'aux_bert.encoder.layer.0.attention.output.LayerNorm.weight', 'aux_bert.encoder.layer.0.attention.output.dense.bias', 'aux_bert.encoder.layer.0.attention.output.dense.weight', 'aux_bert.encoder.layer.0.attention.self.key.bias', 'aux_bert.encoder.layer.0.attention.self.key.weight', 'aux_bert.encoder.layer.0.attention.self.query.bias', 'aux_bert.encoder.layer.0.attention.self.query.weight', 'aux_bert.encoder.layer.0.attention.self.value.bias', 'aux_bert.encoder.layer.0.attention.self.value.weight', 'aux_bert.encoder.layer.0.intermediate.dense.bias', 'a

Some weights of BertModel were not initialized from the model checkpoint at voidism/diffcse-bert-base-uncased-sts and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def render_text(text):
    if POOLING != "prompt_mask":
        return text
    # Matches the author's unsup-bert evaluation template after placeholder expansion.
    # The original evaluator adds a period when the source has no terminal punctuation.
    if text and text[-1] not in '.?"\'':
        text += "."
    return f'This sentence of "{text}" means{tokenizer.mask_token}.'


@torch.inference_mode()
def encode_sentences(texts):
    vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = [render_text(text) for text in texts[start:start + BATCH_SIZE]]
        batch = tokenizer(batch_texts, padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        outputs = model(**batch, return_dict=True)
        if POOLING == "cls_before_pooler":
            embeddings = outputs.last_hidden_state[:, 0, :]
        elif POOLING == "cls_pooler":
            embeddings = outputs.pooler_output
            if embeddings is None:
                raise RuntimeError("This checkpoint has no CLS pooler output")
        elif POOLING == "prompt_mask":
            mask_positions = batch["input_ids"].eq(tokenizer.mask_token_id)
            if not torch.all(mask_positions.sum(dim=1) == 1):
                raise RuntimeError("Prompt template must contain exactly one untruncated [MASK]")
            embeddings = outputs.last_hidden_state[mask_positions]
        else:
            raise ValueError(POOLING)
        vectors.append(F.normalize(embeddings.float(), p=2, dim=1).cpu())
    return torch.cat(vectors).numpy()


# Encode each distinct sentence once. This does not use any gold labels.
unique_sentences = list(dict.fromkeys(frame.sentence1.tolist() + frame.sentence2.tolist()))
print("Distinct sentences to encode:", len(unique_sentences))
vectors = encode_sentences(unique_sentences)
vector_by_sentence = dict(zip(unique_sentences, vectors))
left = np.stack([vector_by_sentence[text] for text in frame.sentence1])
right = np.stack([vector_by_sentence[text] for text in frame.sentence2])
frame["cosine_similarity"] = np.sum(left * right, axis=1)
assert np.isfinite(frame["cosine_similarity"]).all()


Distinct sentences to encode: 5007


In [7]:
gold = frame["gold_score"].to_numpy(dtype=float)
pred = frame["cosine_similarity"].to_numpy(dtype=float)
spearman = spearmanr(gold, pred).statistic
pearson = pearsonr(gold, pred).statistic
summary = pd.DataFrame([{
    "method": METHOD_NAME,
    "dataset": DATASET_NAME,
    "test_pairs": len(frame),
    "spearman": float(spearman),
    "pearson": float(pearson),
    "model_id": MODEL_ID,
    "model_revision": model_sha,
    "dataset_revision": dataset_sha,
    "pooling": POOLING,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}])
display(summary)
print("Examples with low model similarity:")
display(frame.sort_values("cosine_similarity").head(5))
print("Examples with high model similarity:")
display(frame.sort_values("cosine_similarity", ascending=False).head(5))


,method,dataset,test_pairs,spearman,pearson,model_id,model_revision,dataset_revision,pooling,max_length,batch_size,seed
0,DiffCSE,SICKR,4927,0.712216,0.79726,voidism/diffcse-bert-base-uncased-sts,ea605f716bd57f97407640374487c2c9effc94af,b3034994192fae2f41b5937bcf69544e4282fc39,cls_before_pooler,512,32,42


Examples with low model similarity:


,sentence1,sentence2,gold_score,cosine_similarity
937,There is no man sitting on the chair,"The girl, who is little, is combing her hair i...",1.0,0.520445
938,A girl is combing the hair of a little pony's ...,One man is sitting on the chair,1.0,0.528875
2881,A person is riding a motorbike,A woman is slicing potatoes,1.0,0.534901
4874,There is no parrot speaking,A yellow dog is running on white snow on a sun...,1.0,0.563903
2366,A woman is riding on an elephant,A man is opening a soda and drinking it,1.1,0.567604


Examples with high model similarity:


,sentence1,sentence2,gold_score,cosine_similarity
2369,A dog is teasing a monkey at the zoo,A monkey is teasing a dog at the zoo,3.5,0.999211
479,A little boy is jumping into a puddle in the s...,A little boy is jumping into a puddle on the s...,4.9,0.999076
4047,A young man is pushing a motocross bike up a d...,A young man is pushing a motocross bike down a...,3.4,0.998432
673,A boy is spanking a man with a plastic sword,A man is spanking a boy with a plastic sword,3.6,0.998360
4239,A little girl wearing a pink shirt is playing ...,A little girl in a pink shirt is playing t-bal...,4.9,0.998168
